# Week2-1: 6D MCMC from scratch

Now we have the position of all the stars in the 1x1 degree on the sky. \
We assume that each star could belong ot the dwarf galaxy that follows a Gaussian distribution or a the MW forground that follows a flat distribution. \
Now we will determine the 2D distribution of dwarf galaxies on the sky, as well as the member probability of each star. 



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import corner
from glob import glob
import os

In [3]:
data_path = '../data/Dwarfs/'
res_path = '../result/'
fig_path = '../figure/'

files = glob(os.path.join(data_path, "Dwarf*"))
files.sort()
print (np.array(files))



['../data/Dwarfs/Dwarf_00' '../data/Dwarfs/Dwarf_01'
 '../data/Dwarfs/Dwarf_02' '../data/Dwarfs/Dwarf_03'
 '../data/Dwarfs/Dwarf_04' '../data/Dwarfs/Dwarf_05'
 '../data/Dwarfs/Dwarf_06' '../data/Dwarfs/Dwarf_07'
 '../data/Dwarfs/Dwarf_08' '../data/Dwarfs/Dwarf_09'
 '../data/Dwarfs/Dwarf_10' '../data/Dwarfs/Dwarf_11'
 '../data/Dwarfs/Dwarf_12' '../data/Dwarfs/Dwarf_13'
 '../data/Dwarfs/Dwarf_14' '../data/Dwarfs/Dwarf_15'
 '../data/Dwarfs/Dwarf_16' '../data/Dwarfs/Dwarf_17'
 '../data/Dwarfs/Dwarf_18' '../data/Dwarfs/Dwarf_19']


In [4]:
def new_coords(x, y, alpha):

    x_new = x * np.cos(alpha) + y * np.sin(alpha)
    y_new = - x * np.sin(alpha) + y * np.cos(alpha)

    return x_new, y_new

# define the two axis of the ellipse using sigma_x, sigma_y, and alpha (change within pi/2)
def log_Likelihood(theta, x, y):

    xc, yc, sigma_x, sigma_y, alpha, f = theta

    x1, y1 = new_coords(x, y, alpha)
    xc1, yc1 = new_coords(xc, yc, alpha)

    # Calculate the likelihood of being in the dwarf galaxy
    L1 = f/(2*np.pi*sigma_x * sigma_y)*np.exp(-0.5*((x1-xc1)**2./sigma_x**2.+(y1-yc1)**2./sigma_y**2.))
    # Calculate the likelihood of being in the MW foreground
    L2 = (1-f)/10000**2.
    
    logL = np.sum(np.log(L1 + L2))

    return logL

def prob_member(theta, x, y):

    xc, yc, sigma_x, sigma_y, alpha, f = theta

    x1, y1 = new_coords(x, y, alpha)
    xc1, yc1 = new_coords(xc, yc, alpha)

    # Calculate the likelihood of being in the dwarf galaxy
    p1 = 1/(2*np.pi*sigma_x * sigma_y)*np.exp(-0.5*((x1-xc1)**2./sigma_x**2.+(y1-yc1)**2./sigma_y**2.))
    # Calculate the likelihood of being in the MW foreground
    p2 = 1/10000**2.
    
    prob = p1 / (p1 + p2)

    return prob


In [5]:
dwarf_list = np.array([13, 16, 17, 18, 19])
# dwarf_list = np.arange(0, 20, 1)

In [6]:
for k in dwarf_list:

    df = pd.read_csv(files[k])


    xc0 = 0
    yc0 = 0
    sigma_x0 = 100
    sigma_y0 = 100
    alpha0 = 0
    f0=0.1
    param0 = xc0, yc0, sigma_x0, sigma_y0, alpha0, f0

    logL0 = log_Likelihood(param0, df['X_pc'], df['Y_pc'])

    # refine the following steps based on the uncertainties of the results (set steps equal to the uncertainties)
    step_xc = 15
    step_yc = 15
    step_sigma_x = 10
    step_sigma_y = 10
    step_alpha = np.pi/30
    step_f = 0.005

    if k in [2, 13, 16, 17, 18, 19]:

        step_xc = 4
        step_yc = 4
        step_sigma_x = 2
        step_sigma_y = 2
        step_alpha = np.pi/60
        step_f = 0.001

    nsteps = 50000

    samples = np.zeros((nsteps, 8))
    accept = 1


    alpha_min = 0
    alpha_max = np.pi/2

    if k in [0, 10, 11, 15, 17, 18]:

        alpha_min = -np.pi/4
        alpha_max = np.pi/4

    for i in range(nsteps):

        param1 = param0
        logL1 = logL0

        samples[i, :6] = param1
        samples[i, 6] = logL1
        samples[i, 7] = accept

        accept = 0

        xc1, yc1, sigma_x1, sigma_y1, alpha1, f1 = param1
        xc1 = xc1 + np.random.normal(0, step_xc)   
        yc1 = yc1 + np.random.normal(0, step_yc)
        sigma_x1 = sigma_x1 + np.random.normal(0, step_sigma_x)
        sigma_y1 = sigma_y1 + np.random.normal(0, step_sigma_y)
        alpha1 = alpha1 + np.random.normal(0, step_alpha)
        f1 = f1 + np.random.normal(0, step_f)
        param1 = xc1, yc1, sigma_x1, sigma_y1, alpha1, f1


        if (0 < f1 < 1) & (0 <sigma_x1 < 1000) & (0 < sigma_y1 < 1000) & (alpha_min <= alpha1 < alpha_max):
    
            logL1 = log_Likelihood(param1,  df['X_pc'], df['Y_pc'])      

            if logL1 > logL0:

                param0 = param1
                logL0 = logL1
                accept = 1

            else:

                a = np.random.uniform(0, 1)
                if (logL1-logL0) > np.log(a):

                    param0 = param1
                    logL0 = logL1
                    accept = 1

    ind_accept = samples[:, -1] == 1
    print ("Dwarf%02d acceptance rate: %.2f"%(k, len(samples[ind_accept, -1])/nsteps))
    samples =  samples[int(nsteps*0.1):]
    np.savetxt(res_path+'Dwarf_%02d_6D_MCMC_chain.txt'%k, samples)

    fig = corner.corner(
    samples[:,:6],
    labels=[r"xc (pc)", r"yc (pc)", r"$\sigma_x$ (pc)", r"$\sigma_y$ (pc)", r"$\alpha$", r"$f$"],
    show_titles=True,
    title_fmt=".3f",
    quantiles=[0.16, 0.5, 0.84],
    levels=(0.16, 0.5, 0.84),
    color="royalblue",
    fill_contours=True)
    plt.savefig(fig_path+'Dwarf_%02d_6D_MCMC_corner.png'%k, dpi=300)
    plt.close(fig)
    
    param_f = np.zeros(6)
    for i in range(6):
        param_f[i] = np.percentile(samples[:,i], 50)
    print("median parameters:", param_f)

    p_member = np.zeros(len(df))
    p_member = prob_member(param_f, df['X_pc'], df['Y_pc'])
    df['P_member_position'] = p_member  
    df.to_csv(res_path+'Dwarf_%02d_6D_MCMC_prob.csv'%k,index=False)

    fig=plt.figure()
    plt.hist(p_member, bins=50);
    plt.xlabel("Probability of being a member of Dwarf%02d"%1)
    plt.savefig(fig_path+'Dwarf_%02d_6D_MCMC_prob.png'%k, dpi=300)
    plt.close(fig)

    fig=plt.figure()
    plt.scatter(df['X_pc'], df['Y_pc'], c=df['P_member_position'], s=5, cmap='plasma')
    plt.xlabel("X (pc)")
    plt.ylabel("Y (pc)")
    plt.savefig(fig_path+'Dwarf_%02d_6D_MCMC_prob_XY.png'%k, dpi=300)
    plt.close(fig)

    # break



Dwarf13 acceptance rate: 0.28
median parameters: [-5.26229185e+00 -9.00149049e+01  4.78210537e+01  2.16686121e+01
  3.80990797e-01  5.23792169e-02]
Dwarf16 acceptance rate: 0.48
median parameters: [ 3.40184791e+01 -3.90077816e+01  9.62154028e+01  9.34364246e+01
  8.98887106e-01  9.61639091e-02]
Dwarf17 acceptance rate: 0.55
median parameters: [ 1.31409694e+02 -4.24734197e+01  1.03765913e+02  6.53436326e+01
 -4.68489704e-02  5.32322248e-02]
Dwarf18 acceptance rate: 0.51
median parameters: [-1.09804128e+02  1.61653054e+02  1.58267773e+02  9.49128685e+01
 -1.32187880e-01  6.36177706e-02]
Dwarf19 acceptance rate: 0.39
median parameters: [-1.51077889e+02 -1.05609465e+02  6.55172479e+01  3.53234576e+01
  2.97620085e-01  5.45323313e-02]
